# ver3 Option A — Style='Blond' (Latent dim=16)

옵션 A: VAE(변이형 오토인코더; 확률적 압축/복원)도 스타일 데이터만으로 학습
옵션 B: VAE는 전체 데이터(모든 스타일)로 학습 → Diffusion(확산모델; 노이즈→복원 반복)은 선택 스타일(latent)로 학습

**VAE 학습 범위: 스타일 전용(style-only)**

이 노트북은 **스타일 1개(Blond)**만 대상으로 **(화학 X + 관능 Y) 결합 데이터**를 생성하고,
생성 데이터가 (1) 해당 스타일 예측 성능, (2) 전체 예측 성능에 어떤 영향을 주는지 평가합니다.

- 영어/전문용어는 괄호로 짧게 설명합니다.
- GPU는 서버 설정에 맞춰 **2,3번 GPU**를 사용하도록 작성했습니다.
- 데이터 파일: `Supplemental Files and Figure source files.xlsx`


## 0. GPU 설정(필수)
- 이 셀을 **가장 먼저** 실행하세요.
- 이미 `torch`를 import(불러오기)한 상태면, `CUDA_VISIBLE_DEVICES`가 반영되지 않을 수 있으니 **커널(kernel; 실행 엔진) 재시작** 후 다시 실행하세요.


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"  # 서버의 물리 GPU 2,3번만 사용 (노트북 안에서는 0,1로 보임)

In [2]:
# (선택) 실험 실행에 필요한 라이브러리 일괄 설치
# - 이미 설치되어 있다면 이 셀은 건너뛰어도 됩니다.
# - Jupyter 커널이 사용하는 파이썬 환경에 설치됩니다.

import importlib, sys, subprocess

def ensure(pkg: str, import_name: str = None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
        print(f"[OK] {pkg}")
    except ImportError:
        print(f"[INSTALL] {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# 엑셀 로딩(pandas.read_excel)에 필요할 수 있음
ensure("openpyxl", "openpyxl")

# 핵심 과학/ML 패키지
ensure("numpy", "numpy")
ensure("pandas", "pandas")
ensure("scipy", "scipy")
ensure("scikit-learn", "sklearn")
ensure("joblib", "joblib")

# 선택: DOWNSTREAM='xgb'로 바꿀 계획이 있으면 설치 (없으면 주석 처리 가능)
ensure("xgboost", "xgboost")


[OK] openpyxl
[OK] numpy
[OK] pandas
[OK] scipy
[OK] scikit-learn
[OK] joblib
[OK] xgboost


## 1. 라이브러리 불러오기(import; 패키지 로드) & 실험 설정(config; 설정값)
필요 패키지: `torch`, `pandas`, `numpy`, `scikit-learn`, `scipy`

In [3]:
from __future__ import annotations

import json
import math
import random
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, RandomSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.covariance import LedoitWolf
from sklearn.metrics import r2_score, mean_squared_error

from scipy.stats import ks_2samp

# -----------------------
# Reproducibility(재현성)
# -----------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -----------------------
# Paths(경로)
# -----------------------
XLSX_PATH = Path("/home/a202192020/맥주데이터실험/data/Supplemental Files and Figure source files.xlsx")  # 필요 시 절대경로로 수정
assert XLSX_PATH.exists(), f"엑셀 파일을 찾을 수 없습니다: {XLSX_PATH.resolve()}"

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path("runs") / f"ver3_optionA_{RUN_TAG}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------
# Experiment knobs(실험 하이퍼파라미터)
# -----------------------
STYLE_NAME = "Blond"
LATENT_DIM = 16

# VAE training scope: 'style' or 'full'
VAE_SCOPE = "style"

# Training
VAE_EPOCHS = 500          # 작게 시작 후 필요 시 증가
VAE_BATCH_SIZE = 256
VAE_LR = 1e-3
KL_BETA_MAX = 0.1         # KL(정규화) 가중치 최대값 (복원 우선이면 0.01~0.1 권장)
KL_WARMUP_EPOCHS = 100    # beta warmup(서서히 증가)

DIFF_T = 200              # diffusion steps(확산 단계 수). 1000도 가능하지만 느림
DIFF_EPOCHS = 2000        # 데이터가 작아 반복 업데이트가 필요
DIFF_BATCH_SIZE = 2048
DIFF_LR = 2e-4
STEPS_PER_EPOCH = 200     # replacement sampling(복원추출 미니배치)로 한 epoch당 업데이트 횟수

# Sampling(생성)
SYNTH_POOL = 5000         # blond 합성 데이터 pool(저장소) 크기
SAMPLE_BATCH = 4096       # 한 번에 생성할 배치 크기

# Downstream predictor(다운스트림 예측기; 최종 성능 측정 모델)
DOWNSTREAM = "rf"         # 'rf'(RandomForest; 랜덤포레스트) or 'xgb'(XGBoost; 그래디언트 부스팅)
AUG_RATIOS = [0.25, 0.5, 1.0, 2.0]  # blond train 대비 합성 추가 비율

# Device(디바이스)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Visible GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))

CUDA available: True
Visible GPU count: 2
0 NVIDIA A100-PCIE-40GB
1 NVIDIA A100-PCIE-40GB


## 2. 데이터 로드(load; 불러오기) & Blond 스타일 선택
- `Supplementary File S1`: 화학 성분 X
- `Supplementary File S4`: 전문가 관능평가 Y
- 공통 key: `beer_id` (또는 동일한 row order)


In [4]:
# Load sheets
xls = pd.ExcelFile(XLSX_PATH)
df_x = xls.parse("Supplementary File S1")
df_y = xls.parse("Supplementary File S4")

# Sanity check(정합성 검사)
assert (df_x["beer_id"].values == df_y["beer_id"].values).all()
assert (df_x["tasting_category_fine"].values == df_y["tasting_category_fine"].values).all()

meta_cols = ["beer", "beer_id", "tasting_category_fine"]
chem_cols = [c for c in df_x.columns if c not in meta_cols]
sens_cols = [c for c in df_y.columns if c not in meta_cols]

print("N beers:", len(df_x))
print("Chem dims(Dx):", len(chem_cols))
print("Sens dims(Dy):", len(sens_cols))
print("Joint dims(D):", len(chem_cols) + len(sens_cols))

style_counts = df_x["tasting_category_fine"].value_counts()
display(style_counts.head(10))

assert STYLE_NAME in style_counts.index, f"STYLE_NAME='{STYLE_NAME}' not found. Available: {list(style_counts.index)}"

N beers: 250
Chem dims(Dx): 231
Sens dims(Dy): 50
Joint dims(D): 281


tasting_category_fine
Blond           31
Tripel          28
Strong ale      18
Hoppy           17
Stout/Porter    13
Wheat           12
Amber           12
Brown           12
Lambic          11
Pils/Lager      11
Name: count, dtype: int64

## 3. Train/Test split(학습/평가 분할) — style 층화(stratified; 비율 유지)
전체 데이터에서 스타일 비율이 유지되도록 분할합니다.
이후 Blond만 골라서 스타일 내부 평가를 수행합니다.

In [5]:
idx = np.arange(len(df_x))
style = df_x["tasting_category_fine"].astype(str)

idx_train, idx_test = train_test_split(
    idx,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
    stratify=style
)

df_train_x = df_x.iloc[idx_train].reset_index(drop=True)
df_test_x  = df_x.iloc[idx_test].reset_index(drop=True)
df_train_y = df_y.iloc[idx_train].reset_index(drop=True)
df_test_y  = df_y.iloc[idx_test].reset_index(drop=True)

print("Train:", df_train_x.shape, "Test:", df_test_x.shape)
print("Train style counts(top5):")
display(df_train_x["tasting_category_fine"].value_counts().head())

# Blond subset
blond_train_mask = (df_train_x["tasting_category_fine"].astype(str) == STYLE_NAME).values
blond_test_mask  = (df_test_x["tasting_category_fine"].astype(str) == STYLE_NAME).values

df_blond_train_x = df_train_x.loc[blond_train_mask, chem_cols].reset_index(drop=True)
df_blond_train_y = df_train_y.loc[blond_train_mask, sens_cols].reset_index(drop=True)
df_blond_test_x  = df_test_x.loc[blond_test_mask, chem_cols].reset_index(drop=True)
df_blond_test_y  = df_test_y.loc[blond_test_mask, sens_cols].reset_index(drop=True)

print(f"Blond train N={len(df_blond_train_x)}, Blond test N={len(df_blond_test_x)}")

Train: (175, 234) Test: (75, 234)
Train style counts(top5):


tasting_category_fine
Blond           22
Tripel          20
Strong ale      13
Hoppy           12
Stout/Porter     9
Name: count, dtype: int64

Blond train N=22, Blond test N=9


## 4. 전처리(preprocess; 스케일링)
- VAE/Diffusion은 입력 스케일에 민감하므로, **표준화(StandardScaler; 평균 0, 표준편차 1)**를 적용합니다.
- (X,Y) 결합 벡터를 함께 스케일링한 뒤 생성하고, 마지막에 역변환(inverse transform; 원 스케일 복원)합니다.


In [6]:
# Build joint dataframe: [X|Y]  (결측치 처리 포함)
df_train_joint = pd.concat([df_train_x[chem_cols], df_train_y[sens_cols]], axis=1)
df_test_joint  = pd.concat([df_test_x[chem_cols],  df_test_y[sens_cols]],  axis=1)

df_blond_train_joint = pd.concat([df_blond_train_x, df_blond_train_y], axis=1)
df_blond_test_joint  = pd.concat([df_blond_test_x,  df_blond_test_y],  axis=1)

# -----------------------------------------
# 결측치(NaN) 처리: SimpleImputer(mean)
# 이유: S1 화학성분에 일부 NaN이 있어 그대로 학습하면 VAE loss가 NaN으로 붕괴됨
# -----------------------------------------
from sklearn.impute import SimpleImputer
import joblib

# Choose imputer/scaler fit scope for VAE
if VAE_SCOPE == "full":
    imputer_joint = SimpleImputer(strategy="mean").fit(df_train_joint.values)
    joint_fit_arr = imputer_joint.transform(df_train_joint.values)
else:
    imputer_joint = SimpleImputer(strategy="mean").fit(df_blond_train_joint.values)
    joint_fit_arr = imputer_joint.transform(df_blond_train_joint.values)

scaler_joint = StandardScaler().fit(joint_fit_arr)

def joint_transform(df: pd.DataFrame) -> np.ndarray:
    """impute -> scale"""
    arr = imputer_joint.transform(df.values)
    z = scaler_joint.transform(arr).astype(np.float32)
    return z

Z_train_joint_full  = joint_transform(df_train_joint)
Z_test_joint_full   = joint_transform(df_test_joint)
Z_blond_train_joint = joint_transform(df_blond_train_joint)
Z_blond_test_joint  = joint_transform(df_blond_test_joint)

input_dim = Z_train_joint_full.shape[1]
print("Scaled joint dim:", input_dim)

# Finite check(유한값 검사)
assert np.isfinite(Z_blond_train_joint).all(), "Z_blond_train_joint에 NaN/Inf가 있습니다. 전처리를 확인하세요."

# Save preprocessors
joblib.dump(imputer_joint, OUT_DIR / "imputer_joint.joblib")
joblib.dump(scaler_joint,  OUT_DIR / "scaler_joint.joblib")


Scaled joint dim: 281


['runs/ver3_optionA_20260128_034403/scaler_joint.joblib']

## 5. PyTorch Dataset(데이터셋) & DataLoader(미니배치 로더)
- 작은 데이터에서 GPU 사용률을 높이기 위해 `RandomSampler(replacement=True)`로 반복 샘플링합니다.
- 단, 통계적 정보는 늘지 않으므로 과적합(overfitting; 외우기) 가능성이 큽니다. 그래서 **평가(test; 테스트)는 항상 real(원본)로만** 합니다.


In [7]:
class ArrayDataset(Dataset):
    def __init__(self, arr: np.ndarray):
        self.arr = torch.from_numpy(arr).float()
    def __len__(self):
        return self.arr.shape[0]
    def __getitem__(self, idx):
        return self.arr[idx]

# VAE train dataset
if VAE_SCOPE == "full":
    ds_vae = ArrayDataset(Z_train_joint_full)
else:
    ds_vae = ArrayDataset(Z_blond_train_joint)

# small val split
val_size = max(1, int(0.1 * len(ds_vae)))
train_size = len(ds_vae) - val_size
ds_vae_train, ds_vae_val = random_split(ds_vae, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

# Replacement sampler to increase updates
sampler = RandomSampler(ds_vae_train, replacement=True, num_samples=STEPS_PER_EPOCH * VAE_BATCH_SIZE)

dl_vae_train = DataLoader(ds_vae_train, batch_size=VAE_BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
dl_vae_val   = DataLoader(ds_vae_val, batch_size=min(VAE_BATCH_SIZE, len(ds_vae_val)), shuffle=False, num_workers=2, pin_memory=True)

print("VAE train steps/epoch:", len(dl_vae_train), "val batches:", len(dl_vae_val))

VAE train steps/epoch: 200 val batches: 1


## 6. VAE 구현(Implementation)
- Encoder(인코더): joint 벡터 → 잠재벡터 z의 평균/분산
- Decoder(디코더): z → joint 벡터 복원
- Loss(손실): 복원 손실(reconstruction loss; MSE) + KL(정규화) 손실(KL divergence; 잠재분포 규제)
- `beta warmup`: 초반엔 복원에 집중, 점진적으로 KL 가중치 증가


In [8]:
class MLPVAE(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int, hidden=(512, 256, 256)):
        super().__init__()
        h1, h2, h3 = hidden
        self.enc = nn.Sequential(
            nn.Linear(input_dim, h1), nn.SiLU(),
            nn.Linear(h1, h2), nn.SiLU(),
            nn.Linear(h2, h3), nn.SiLU(),
        )
        self.mu = nn.Linear(h3, latent_dim)
        self.logvar = nn.Linear(h3, latent_dim)

        self.dec = nn.Sequential(
            nn.Linear(latent_dim, h3), nn.SiLU(),
            nn.Linear(h3, h2), nn.SiLU(),
            nn.Linear(h2, h1), nn.SiLU(),
            nn.Linear(h1, input_dim),
        )

    def encode(self, x):
        h = self.enc(x)
        return self.mu(h), self.logvar(h)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

def kl_beta(epoch: int, beta_max: float, warmup_epochs: int) -> float:
    # linear warmup
    if warmup_epochs <= 0:
        return beta_max
    return float(beta_max * min(1.0, epoch / warmup_epochs))

vae = MLPVAE(input_dim=input_dim, latent_dim=LATENT_DIM).to(device)

# Multi-GPU (DataParallel; 간단한 멀티GPU 래퍼)
if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    vae = nn.DataParallel(vae)  # uses visible GPUs 0,1 (== physical 2,3)
    print("Using DataParallel on", torch.cuda.device_count(), "GPUs")

opt = torch.optim.AdamW(vae.parameters(), lr=VAE_LR, weight_decay=1e-6)
scaler_amp = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

Using DataParallel on 2 GPUs


/tmp/ipykernel_2682664/1447220460.py:52: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_amp = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


## 7. VAE 학습(training; 훈련)
- Early stopping(조기 종료): validation(검증) 복원 손실이 더 이상 줄지 않으면 종료
- 모델/로그는 `runs/ver3_option.../` 폴더에 저장됩니다.


In [ ]:
def vae_loss(x, recon, mu, logvar, beta: float):
    # recon MSE per-dimension 평균
    recon_loss = F.mse_loss(recon, x, reduction="mean")
    # KL divergence 평균
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl, recon_loss.detach(), kl.detach()

# -----------------------
# AMP / TF32 설정 (A100 권장)
# -----------------------
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

use_cuda = torch.cuda.is_available()
use_bf16 = bool(use_cuda and hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported())
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
scaler_amp = torch.cuda.amp.GradScaler(enabled=(use_cuda and not use_bf16))

print("[VAE AMP] cuda:", use_cuda, "bf16:", use_bf16, "dtype:", amp_dtype, "scaler:", scaler_amp.is_enabled())

best_val = float("inf")
best_state = None
patience = 50
bad = 0
history = []

for epoch in range(1, VAE_EPOCHS + 1):
    vae.train()
    beta = kl_beta(epoch, KL_BETA_MAX, KL_WARMUP_EPOCHS)

    tr_losses, tr_recon, tr_kl = [], [], []

    for xb in dl_vae_train:
        xb = xb.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_cuda):
            recon, mu, logvar, _ = vae(xb)
            # logvar 폭발 방지(중요)
            logvar = torch.clamp(logvar, -10.0, 10.0)
            loss, rloss, kl = vae_loss(xb, recon, mu, logvar, beta)

        if not torch.isfinite(loss):
            # NaN/Inf 발생 시 업데이트 스킵 (데이터/학습률/모델을 점검해야 함)
            continue

        if scaler_amp.is_enabled():
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
            scaler_amp.step(opt)
            scaler_amp.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(vae.parameters(), 1.0)
            opt.step()

        tr_losses.append(float(loss.detach().cpu()))
        tr_recon.append(float(rloss.detach().cpu()))
        tr_kl.append(float(kl.detach().cpu()))

    # validation (전체 평균)
    vae.eval()
    val_losses, val_recons, val_kls = [], [], []
    with torch.no_grad():
        for xb in dl_vae_val:
            xb = xb.to(device, non_blocking=True)
            with torch.amp.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_cuda):
                recon, mu, logvar, _ = vae(xb)
                logvar = torch.clamp(logvar, -10.0, 10.0)
                loss, rloss, kl = vae_loss(xb, recon, mu, logvar, beta=KL_BETA_MAX)

            if torch.isfinite(loss):
                val_losses.append(float(loss.detach().cpu()))
                val_recons.append(float(rloss.detach().cpu()))
                val_kls.append(float(kl.detach().cpu()))

    if len(val_recons) == 0:
        raise RuntimeError("Validation에서 finite loss가 0개입니다. 입력 NaN/Inf 또는 학습 발산 가능성이 큽니다.")

    history.append({
        "epoch": epoch,
        "beta": float(beta),
        "train_loss": float(np.mean(tr_losses)) if tr_losses else float("nan"),
        "train_recon": float(np.mean(tr_recon)) if tr_recon else float("nan"),
        "train_kl": float(np.mean(tr_kl)) if tr_kl else float("nan"),
        "val_loss": float(np.mean(val_losses)),
        "val_recon": float(np.mean(val_recons)),
        "val_kl": float(np.mean(val_kls)),
    })

    if epoch % 25 == 0 or epoch == 1:
        h = history[-1]
        print(f"[VAE] ep={epoch:4d} beta={h['beta']:.4f} "
              f"train_recon={h['train_recon']:.4f} val_recon={h['val_recon']:.4f} "
              f"train_kl={h['train_kl']:.4f} val_kl={h['val_kl']:.4f}")

    # early stop on val_recon (복원 중심)
    score = history[-1]["val_recon"]
    if score < best_val - 1e-6:
        best_val = score
        bad = 0
        module = vae.module if isinstance(vae, nn.DataParallel) else vae
        best_state = {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}
    else:
        bad += 1
        if bad >= patience:
            print("[VAE] Early stopping triggered.")
            break

# restore best (안전장치 포함)
module = vae.module if isinstance(vae, nn.DataParallel) else vae
if best_state is None:
    print("[WARN] best_state=None (모든 epoch에서 score 갱신 실패). 현재 state로 진행합니다.")
    best_state = {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}

module.load_state_dict(best_state)

# save
torch.save(best_state, OUT_DIR / "vae_state_dict.pt")
pd.DataFrame(history).to_csv(OUT_DIR / "vae_training_log.csv", index=False)

print("Best val recon:", best_val)


[VAE AMP] cuda: True bf16: True dtype: torch.bfloat16 scaler: False


/tmp/ipykernel_2682664/3802452541.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_amp = torch.cuda.amp.GradScaler(enabled=(use_cuda and not use_bf16))


[VAE] ep=   1 beta=0.0010 train_recon=0.1503 val_recon=2.3366 train_kl=4.6499 val_kl=0.8369


## 8. Blond 잠재벡터(latent vector; z) 추출
- Diffusion(확산모델)은 **Blond train**의 z만 사용해서 학습합니다.
- Option B에서도 동일하게 Blond z만으로 diffusion을 학습합니다.


In [ ]:
module = vae.module if isinstance(vae, nn.DataParallel) else vae
module.eval()

@torch.no_grad()
def encode_np(arr_scaled: np.ndarray) -> np.ndarray:
    x = torch.from_numpy(arr_scaled).float().to(device)
    mu, logvar = module.encode(x)
    z = mu  # diffusion에는 평균(mu)만 사용 (안정적)
    return z.detach().cpu().numpy()

z_blond_train = encode_np(Z_blond_train_joint)
print("z_blond_train:", z_blond_train.shape)

# Diffusion dataset
ds_diff = ArrayDataset(z_blond_train.astype(np.float32))
sampler_diff = RandomSampler(ds_diff, replacement=True, num_samples=STEPS_PER_EPOCH * DIFF_BATCH_SIZE)
dl_diff = DataLoader(ds_diff, batch_size=DIFF_BATCH_SIZE, sampler=sampler_diff, num_workers=4, pin_memory=True)

## 9. Latent Diffusion(잠재확산) 모델 구현
- DDPM(denoising diffusion probabilistic model; 디노이징 확산 확률 모델) 형태
- 입력: z_t(노이즈가 섞인 잠재벡터) + t(시간 step)
- 출력: ε(노이즈) 예측


In [ ]:
# --- diffusion utilities ---
def linear_beta_schedule(T, beta_start=1e-4, beta_end=2e-2):
    return torch.linspace(beta_start, beta_end, T)

class SinusoidalTimeEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        # t: (B,) int64
        device = t.device
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=device) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0,1))
        return emb

class DiffusionMLP(nn.Module):
    def __init__(self, latent_dim, time_dim=128, hidden=512):
        super().__init__()
        self.time_emb = SinusoidalTimeEmb(time_dim)
        self.net = nn.Sequential(
            nn.Linear(latent_dim + time_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, latent_dim),
        )
    def forward(self, zt, t):
        te = self.time_emb(t)
        x = torch.cat([zt, te], dim=1)
        return self.net(x)

T = DIFF_T
betas = linear_beta_schedule(T).to(device)  # (T,)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), alphas_cumprod[:-1]], dim=0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

def q_sample(z0, t, noise):
    # z_t = sqrt(a_bar)*z0 + sqrt(1-a_bar)*noise
    a = sqrt_alphas_cumprod[t].unsqueeze(1)
    b = sqrt_one_minus_alphas_cumprod[t].unsqueeze(1)
    return a * z0 + b * noise

diff_model = DiffusionMLP(latent_dim=LATENT_DIM).to(device)
if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    diff_model = nn.DataParallel(diff_model)

opt_d = torch.optim.AdamW(diff_model.parameters(), lr=DIFF_LR, weight_decay=1e-6)
scaler_d = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

## 10. Diffusion 학습(training)
- 목표: ε 예측 MSE 최소화
- 데이터가 작으므로 `replacement sampling`으로 충분히 업데이트 횟수를 확보합니다.


In [ ]:
diff_model.train()
dlog = []

for epoch in range(1, DIFF_EPOCHS + 1):
    losses = []
    for z0 in dl_diff:
        z0 = z0.to(device, non_blocking=True)
        bsz = z0.size(0)
        t = torch.randint(0, T, (bsz,), device=device).long()
        noise = torch.randn_like(z0)
        zt = q_sample(z0, t, noise)

        opt_d.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            eps_pred = diff_model(zt, t)
            loss = F.mse_loss(eps_pred, noise)
        scaler_d.scale(loss).backward()
        scaler_d.step(opt_d)
        scaler_d.update()
        losses.append(loss.item())

    m = float(np.mean(losses))
    dlog.append({"epoch": epoch, "loss": m})

    if epoch % 200 == 0 or epoch == 1:
        print(f"[DIFF] ep={epoch:4d} loss={m:.6f}")

    # simple early stop heuristic
    if epoch >= 600 and m < 0.02:
        # 데이터가 매우 작으면 빠르게 내려갈 수 있음
        pass

# save
module_d = diff_model.module if isinstance(diff_model, nn.DataParallel) else diff_model
torch.save(module_d.state_dict(), OUT_DIR / "diff_state_dict.pt")
pd.DataFrame(dlog).to_csv(OUT_DIR / "diff_training_log.csv", index=False)

## 11. Blond 합성 데이터 생성(sampling; 샘플링)
- latent z를 diffusion으로 생성
- VAE decoder로 (X,Y) 복원 → scaler inverse transform(역변환)
- 결과: Blond 합성 (화학 X + 관능 Y) 데이터


In [ ]:
module_d = diff_model.module if isinstance(diff_model, nn.DataParallel) else diff_model
module_d.eval()
module.eval()

@torch.no_grad()
def p_sample_step(zt, t):
    # predict noise
    eps = module_d(zt, t)
    beta_t = betas[t].unsqueeze(1)
    alpha_t = alphas[t].unsqueeze(1)
    a_bar = alphas_cumprod[t].unsqueeze(1)

    # DDPM mean
    mean = (1/torch.sqrt(alpha_t)) * (zt - (beta_t/torch.sqrt(1-a_bar)) * eps)
    if (t[0].item() == 0):
        return mean
    var = posterior_variance[t].unsqueeze(1)
    noise = torch.randn_like(zt)
    return mean + torch.sqrt(var) * noise

@torch.no_grad()
def sample_latents(n: int, batch: int):
    out = []
    n_done = 0
    while n_done < n:
        cur = min(batch, n - n_done)
        zt = torch.randn(cur, LATENT_DIM, device=device)
        # reverse process
        for step in reversed(range(T)):
            t = torch.full((cur,), step, device=device, dtype=torch.long)
            zt = p_sample_step(zt, t)
        out.append(zt.detach().cpu().numpy())
        n_done += cur
    return np.concatenate(out, axis=0)

z_synth = sample_latents(SYNTH_POOL, SAMPLE_BATCH).astype(np.float32)
print("z_synth:", z_synth.shape)

@torch.no_grad()
def decode_latents(z_np: np.ndarray) -> np.ndarray:
    z = torch.from_numpy(z_np).float().to(device)
    xhat = module.decode(z)
    return xhat.detach().cpu().numpy()

joint_synth_scaled = decode_latents(z_synth).astype(np.float32)
joint_synth = scaler_joint.inverse_transform(joint_synth_scaled).astype(np.float32)

# split back to X/Y
Xs = joint_synth[:, :len(chem_cols)]
Ys = joint_synth[:, len(chem_cols):]

df_synth_x = pd.DataFrame(Xs, columns=chem_cols)
df_synth_y = pd.DataFrame(Ys, columns=sens_cols)

df_synth_x.to_csv(OUT_DIR / "blond_synth_X.csv", index=False)
df_synth_y.to_csv(OUT_DIR / "blond_synth_Y.csv", index=False)

print("Saved synth X/Y to", OUT_DIR)

## 12. 생성 품질(Fidelity; 원본 유사성) — Blond 내부
- KS-test(분포 유사도 검사): 변수별 분포가 비슷한가?
- PCD(상관 구조 차이): X–X, X–Y 상관(관계)이 유지되는가?
- DCR(최근접 거리): 합성이 너무 복제/너무 이상치인지?

⚠️ 주의: Blond 샘플 수가 작으면(예: 20개 안팎), 상관 추정이 불안정합니다 → LedoitWolf shrinkage(축소 공분산; 안정화) 사용


In [ ]:
def ks_summary(real: pd.DataFrame, synth: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for c in real.columns:
        r = real[c].values
        s = synth[c].values
        # NaN guard(결측치 제거 후 KS)
        r = r[np.isfinite(r)]
        s = s[np.isfinite(s)]
        if len(r) < 2 or len(s) < 2:
            continue
        stat, p = ks_2samp(r, s)
        rows.append({"col": c, "ks": float(stat), "p": float(p)})
    df = pd.DataFrame(rows).sort_values("ks", ascending=False)
    return df

def corr_shrinkage(arr: np.ndarray) -> np.ndarray:
    # Ledoit-Wolf shrinkage(공분산 shrinkage: 샘플이 적을 때 공분산을 안정화)
    lw = LedoitWolf().fit(arr)
    cov = lw.covariance_
    d = np.sqrt(np.diag(cov))
    corr = cov / (d[:,None] * d[None,:] + 1e-12)
    corr = np.clip(corr, -1, 1)
    return corr

def pcd_xx(real_x: np.ndarray, synth_x: np.ndarray) -> float:
    c1 = corr_shrinkage(real_x)
    c2 = corr_shrinkage(synth_x)
    return float(np.linalg.norm(c1 - c2, ord="fro") / c1.size)

def pcd_xy(real_joint: np.ndarray, synth_joint: np.ndarray, dx: int) -> float:
    # compute shrinkage corr on full joint then take X-Y block
    c1 = corr_shrinkage(real_joint)
    c2 = corr_shrinkage(synth_joint)
    block1 = c1[:dx, dx:]
    block2 = c2[:dx, dx:]
    return float(np.linalg.norm(block1 - block2, ord="fro") / block1.size)

def dcr(real_joint_scaled: np.ndarray, synth_joint_scaled: np.ndarray) -> np.ndarray:
    # nearest distance in scaled space
    nn = NearestNeighbors(n_neighbors=1, algorithm="auto").fit(real_joint_scaled)
    dist, _ = nn.kneighbors(synth_joint_scaled)
    return dist.ravel()

# Real blond (train) vs synth pool
df_synth_joint = pd.concat([df_synth_x, df_synth_y], axis=1)

# KS는 column별로 NaN 제거해서 계산하므로 원본 그대로 사용
ks_df = ks_summary(df_blond_train_joint, df_synth_joint)
ks_df.to_csv(OUT_DIR / "fidelity_blond_ks.csv", index=False)

# PCD / DCR은 결측치가 있으면 깨지므로, 학습에서 쓴 mean-imputer로 결측치를 채운 후 계산
from sklearn.impute import SimpleImputer

imputer_x = SimpleImputer(strategy="mean").fit(df_blond_train_x.values)
real_x_imp = imputer_x.transform(df_blond_train_x.values).astype(np.float32)
synth_x_imp = imputer_x.transform(df_synth_x.values).astype(np.float32)

real_joint_imp  = imputer_joint.transform(df_blond_train_joint.values).astype(np.float32)
synth_joint_imp = imputer_joint.transform(df_synth_joint.values).astype(np.float32)

pcd_xx_val = pcd_xx(real_x_imp, synth_x_imp)
pcd_xy_val = pcd_xy(real_joint_imp, synth_joint_imp, dx=len(chem_cols))

# DCR은 scaled space에서 계산 (impute -> scale)
synth_scaled = scaler_joint.transform(synth_joint_imp).astype(np.float32)
dcr_vals = dcr(Z_blond_train_joint, synth_scaled)

fidelity = {
    "pcd_xx": pcd_xx_val,
    "pcd_xy": pcd_xy_val,
    "dcr_mean": float(dcr_vals.mean()),
    "dcr_median": float(np.median(dcr_vals)),
    "dcr_p05": float(np.quantile(dcr_vals, 0.05)),
    "dcr_p95": float(np.quantile(dcr_vals, 0.95)),
}

with open(OUT_DIR / "fidelity_blond_summary.json", "w", encoding="utf-8") as f:
    json.dump(fidelity, f, ensure_ascii=False, indent=2)

print("Fidelity summary:", fidelity)
display(ks_df.head(10))


## 13. Downstream predictor(다운스트림 예측기) 평가
### 평가 시나리오
1) Blond 내부: (Blond real) vs (Blond real + Blond synth)
2) 전체: (All real) vs (All real + Blond synth)

지표(metric; 평가값):
- R²(결정계수; 설명력) ↑ 좋음
- RMSE(평균제곱근오차; 오차 크기) ↓ 좋음


In [ ]:
# RMSE helper (sklearn 버전 호환)
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def eval_multioutput(y_true: np.ndarray, y_pred: np.ndarray):
    r2 = float(r2_score(y_true, y_pred, multioutput="uniform_average"))
    rm = rmse(y_true, y_pred)
    return r2, rm

def make_downstream_model(kind: str):
    kind = kind.lower()
    if kind == "rf":
        from sklearn.ensemble import RandomForestRegressor
        # RF는 multi-output을 직접 지원
        return RandomForestRegressor(
            n_estimators=500,
            random_state=SEED,
            n_jobs=-1,
            max_features="sqrt"
        )
    elif kind == "xgb":
        try:
            from xgboost import XGBRegressor
            from sklearn.multioutput import MultiOutputRegressor
            base = XGBRegressor(
                n_estimators=1500,
                learning_rate=0.03,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=1.0,
                objective="reg:squarederror",
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1,
            )
            return MultiOutputRegressor(base, n_jobs=-1)
        except Exception as e:
            print("XGBoost not available, fallback to RF. Error:", e)
            return make_downstream_model("rf")
    else:
        raise ValueError("DOWNSTREAM must be 'rf' or 'xgb'")

def fit_predict(model, Xtr, Ytr, Xte):
    model.fit(Xtr, Ytr)
    return model.predict(Xte)

# -------------------------
# Prepare data arrays
# -------------------------
X_all_tr = df_train_x[chem_cols].values.astype(np.float32)
Y_all_tr = df_train_y[sens_cols].values.astype(np.float32)
X_all_te = df_test_x[chem_cols].values.astype(np.float32)
Y_all_te = df_test_y[sens_cols].values.astype(np.float32)

X_bl_tr = df_blond_train_x.values.astype(np.float32)
Y_bl_tr = df_blond_train_y.values.astype(np.float32)
X_bl_te = df_blond_test_x.values.astype(np.float32)
Y_bl_te = df_blond_test_y.values.astype(np.float32)

# synth pool arrays
Xs_pool = df_synth_x.values.astype(np.float32)
Ys_pool = df_synth_y.values.astype(np.float32)

# -----------------------------------------
# 결측치 처리(중요): 화학 X에 NaN이 있음 → 모델 학습이 깨짐
# -----------------------------------------
from sklearn.impute import SimpleImputer
impX = SimpleImputer(strategy="mean").fit(X_all_tr)  # fit on real train only (데이터 누수 방지)

X_all_tr_i = impX.transform(X_all_tr)
X_all_te_i = impX.transform(X_all_te)
X_bl_tr_i  = impX.transform(X_bl_tr)
X_bl_te_i  = impX.transform(X_bl_te)
Xs_pool_i  = impX.transform(Xs_pool)

# Standardize X only for downstream (fit on real train to avoid leakage)
sx = StandardScaler().fit(X_all_tr_i)
X_all_tr_s = sx.transform(X_all_tr_i)
X_all_te_s = sx.transform(X_all_te_i)
X_bl_tr_s  = sx.transform(X_bl_tr_i)
X_bl_te_s  = sx.transform(X_bl_te_i)
Xs_pool_s  = sx.transform(Xs_pool_i)

# -------------------------
# 13-1) Blond-only baseline vs AUG
# -------------------------
results = []

# baseline (TRTR within blond)
model = make_downstream_model(DOWNSTREAM)
pred = fit_predict(model, X_bl_tr_s, Y_bl_tr, X_bl_te_s)
r2_base, rm_base = eval_multioutput(Y_bl_te, pred)
results.append({
    "scenario": "blond_TRTR",
    "aug_ratio": 0.0,
    "r2": r2_base,
    "rmse": rm_base,
    "delta_r2_vs_blond_TRTR": 0.0,
    "delta_rmse_vs_blond_TRTR": 0.0,
})
print("[Blond TRTR] R2=", r2_base, "RMSE=", rm_base)

rng = np.random.RandomState(SEED)
for r in AUG_RATIOS:
    n_aug = int(len(X_bl_tr) * r)
    idx_aug = rng.choice(len(Xs_pool_s), size=n_aug, replace=False)

    X_aug = np.vstack([X_bl_tr_s, Xs_pool_s[idx_aug]])
    Y_aug = np.vstack([Y_bl_tr, Ys_pool[idx_aug]])

    model = make_downstream_model(DOWNSTREAM)
    pred = fit_predict(model, X_aug, Y_aug, X_bl_te_s)
    r2, rm = eval_multioutput(Y_bl_te, pred)
    results.append({
        "scenario": "blond_AUG",
        "aug_ratio": r,
        "n_aug": n_aug,
        "r2": r2,
        "rmse": rm,
        "delta_r2_vs_blond_TRTR": r2 - r2_base,
        "delta_rmse_vs_blond_TRTR": rm - rm_base,
    })
    print(f"[Blond AUG r={r}] R2={r2:.4f} (Δ{r2-r2_base:+.4f}) RMSE={rm:.4f} (Δ{rm-rm_base:+.4f})")

# -------------------------
# 13-2) Full baseline vs (Full + Blond synth)
# -------------------------
model = make_downstream_model(DOWNSTREAM)
pred = fit_predict(model, X_all_tr_s, Y_all_tr, X_all_te_s)
r2_full, rm_full = eval_multioutput(Y_all_te, pred)
results.append({
    "scenario": "full_TRTR",
    "aug_ratio": 0.0,
    "r2": r2_full,
    "rmse": rm_full,
    "delta_r2_vs_full_TRTR": 0.0,
    "delta_rmse_vs_full_TRTR": 0.0,
})
print("[Full TRTR] R2=", r2_full, "RMSE=", rm_full)

for r in AUG_RATIOS:
    n_aug = int(len(X_bl_tr) * r)  # keep same scale as blond train
    idx_aug = rng.choice(len(Xs_pool_s), size=n_aug, replace=False)

    X_aug = np.vstack([X_all_tr_s, Xs_pool_s[idx_aug]])
    Y_aug = np.vstack([Y_all_tr, Ys_pool[idx_aug]])

    model = make_downstream_model(DOWNSTREAM)
    pred = fit_predict(model, X_aug, Y_aug, X_all_te_s)
    r2, rm = eval_multioutput(Y_all_te, pred)

    # also blond-only performance inside full test
    blond_te_mask = (df_test_x["tasting_category_fine"].astype(str) == STYLE_NAME).values
    r2_bl, rm_bl = eval_multioutput(Y_all_te[blond_te_mask], pred[blond_te_mask])

    results.append({
        "scenario": "full_AUG+blondSynth",
        "aug_ratio": r,
        "n_aug": n_aug,
        "r2": r2,
        "rmse": rm,
        "delta_r2_vs_full_TRTR": r2 - r2_full,
        "delta_rmse_vs_full_TRTR": rm - rm_full,
        "r2_blond_test": r2_bl,
        "rmse_blond_test": rm_bl,
    })
    print(f"[Full+AUG(r={r})] R2={r2:.4f} (Δ{r2-r2_full:+.4f}) RMSE={rm:.4f} (Δ{rm-rm_full:+.4f}) | Blond-test R2={r2_bl:.4f}")

df_res = pd.DataFrame(results)
df_res.to_csv(OUT_DIR / "downstream_results.csv", index=False)
display(df_res)


## 14. 설정(config) 저장 & 마무리
- 학습된 모델: `vae_state_dict.pt`, `diff_state_dict.pt`
- 생성 데이터: `blond_synth_X.csv`, `blond_synth_Y.csv`
- 결과: `downstream_results.csv`, `fidelity_blond_*`


In [ ]:
config = {
    "STYLE_NAME": STYLE_NAME,
    "VAE_SCOPE": VAE_SCOPE,
    "LATENT_DIM": LATENT_DIM,
    "VAE_EPOCHS": VAE_EPOCHS,
    "DIFF_T": DIFF_T,
    "DIFF_EPOCHS": DIFF_EPOCHS,
    "SYNTH_POOL": SYNTH_POOL,
    "DOWNSTREAM": DOWNSTREAM,
    "AUG_RATIOS": AUG_RATIOS,
    "SEED": SEED,
}
with open(OUT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Done. Outputs saved to:", OUT_DIR.resolve())